In [6]:
"""
Veloura Synthetic Dataset Generator
Fashion & Beauty Ecommerce (India) - Realistic messy dataset for portfolio project
Timeline: 2023-01-01 to 2026-09-15
"""
import random
import string
from datetime import date, timedelta, datetime
import pandas as pd
import numpy as np
from faker import Faker

fake = Faker('en_IN')
Faker.seed(42)
random.seed(42)
np.random.seed(42)

START_DATE = date(2023, 1, 1)
END_DATE = date(2026, 9, 15)
TOTAL_DAYS = (END_DATE - START_DATE).days

INDIAN_STATES = [
    "Maharashtra", "Delhi", "Karnataka", "Tamil Nadu", "West Bengal", "Gujarat",
    "Rajasthan", "Uttar Pradesh", "Telangana", "Punjab", "Kerala", "Haryana",
    "Madhya Pradesh", "Bihar", "Odisha", "Assam"
]
STATE_VARIANTS = {  # for messiness injection later
    "Delhi": ["Delhi", "New Delhi", "DELHI", "delhi"],
    "Maharashtra": ["Maharashtra", "MAHARASHTRA", "maharashtra"],
}

# ---------------------------------------------------------------------------
# 1. PRODUCTS  (55 products, 5 categories, ~11 each)
# ---------------------------------------------------------------------------
CATEGORIES = {
    "Clothing": {
        "items": ["Kurti", "Saree", "T-Shirt", "Jeans", "Ethnic Gown", "Palazzo Set",
                  "Blazer", "Co-ord Set", "Dress", "Jacket", "Trousers"],
        "brands": ["Veloura Basics", "Zorra", "Anokhi Threads", "Urban Weft"],
        "return_rate": (0.18, 0.25),
        "price_range": (699, 3499),
    },
    "Footwear": {
        "items": ["Sneakers", "Sandals", "Heels", "Flats", "Loafers", "Flip Flops",
                  "Boots", "Wedges", "Sports Shoes", "Juttis", "Slippers"],
        "brands": ["StepStyle", "Veloura Walk", "TrendSole", "ComfortLine"],
        "return_rate": (0.15, 0.20),
        "price_range": (499, 2999),
    },
    "Hats": {
        "items": ["Sun Hat", "Cap", "Beanie", "Fedora", "Bucket Hat", "Beret",
                  "Visor", "Straw Hat", "Winter Hat", "Snapback", "Panama Hat"],
        "brands": ["Veloura Accessories", "CapCraft", "HeadTurn"],
        "return_rate": (0.05, 0.08),
        "price_range": (299, 999),
    },
    "Makeup": {
        "items": ["Lipstick", "Foundation", "Kajal", "Compact Powder", "Blush",
                  "Eyeliner", "Mascara", "Highlighter", "Lip Gloss", "Concealer", "Primer"],
        "brands": ["GlowUp", "Veloura Beauty", "PureTone", "ColorPop India"],
        "return_rate": (0.08, 0.12),
        "price_range": (199, 1299),
    },
    "Soap": {
        "items": ["Herbal Soap", "Charcoal Soap", "Sandalwood Soap", "Rose Soap",
                  "Neem Soap", "Aloe Vera Soap", "Goat Milk Soap", "Turmeric Soap",
                  "Coffee Soap", "Lavender Soap", "Oatmeal Soap"],
        "brands": ["NatureCare", "Veloura Essentials", "HerbNest"],
        "return_rate": (0.03, 0.05),
        "price_range": (99, 399),
    },
}

def gen_products():
    rows = []
    pid = 1
    for cat, info in CATEGORIES.items():
        for item in info["items"]:
            cost = round(random.uniform(*info["price_range"]) * 0.55, 2)
            selling = round(cost / 0.55, 2)
            launch_offset = random.randint(0, int(TOTAL_DAYS * 0.6))  # most launched early-mid
            launch_date = START_DATE + timedelta(days=launch_offset)
            rows.append({
                "product_id": pid,
                "product_name": f"{item} - {fake.word().capitalize()}",
                "category": cat,
                "brand": random.choice(info["brands"]),
                "cost_price": cost,
                "selling_price": selling,
                "stock_quantity": random.randint(20, 500),
                "launch_date": launch_date.isoformat(),
                "_return_rate": random.uniform(*info["return_rate"]),  # internal use only
            })
            pid += 1
    return pd.DataFrame(rows)

products_df = gen_products()
print(f"Products generated: {len(products_df)}")

# ---------------------------------------------------------------------------
# 2. PROMOTIONS  (~100, festival-anchored + generic)
# ---------------------------------------------------------------------------
FESTIVALS = [
    ("New Year Sale", "01-01", 7, (10, 20)),
    ("Republic Day Sale", "01-26", 5, (15, 30)),
    ("Holi Splash Offer", "03-15", 6, (10, 25)),
    ("Summer EOSS", "06-15", 14, (30, 50)),
    ("Independence Day Sale", "08-15", 5, (15, 35)),
    ("Rakhi Special", "08-25", 4, (10, 20)),
    ("Navratri Fest Offer", "10-05", 9, (15, 30)),
    ("Diwali Dhamaka", "11-01", 10, (25, 50)),
    ("Winter EOSS", "12-20", 12, (30, 50)),
]

def gen_promotions():
    rows = []
    promo_id = 1
    for year in range(2023, 2027):
        for name, md, duration, disc_range in FESTIVALS:
            month, day = map(int, md.split("-"))
            try:
                start = date(year, month, day)
            except ValueError:
                continue
            if start < START_DATE or start > END_DATE:
                continue
            end = start + timedelta(days=duration)
            rows.append({
                "promotion_id": promo_id,
                "promotion_name": f"{name} {year}",
                "promotion_type": "Festival",
                "discount_percentage": round(random.uniform(*disc_range), 2),
                "start_date": start.isoformat(),
                "end_date": end.isoformat(),
            })
            promo_id += 1
    # top up with generic promos to reach ~100
    generic_types = ["Flash Sale", "Clearance", "Welcome Offer", "Weekend Deal", "Category Special"]
    while len(rows) < 100:
        offset = random.randint(0, TOTAL_DAYS - 10)
        start = START_DATE + timedelta(days=offset)
        end = start + timedelta(days=random.randint(3, 10))
        rows.append({
            "promotion_id": promo_id,
            "promotion_name": f"{random.choice(generic_types)} {start.strftime('%b%Y')}",
            "promotion_type": random.choice(generic_types),
            "discount_percentage": round(random.uniform(5, 40), 2),
            "start_date": start.isoformat(),
            "end_date": end.isoformat(),
        })
        promo_id += 1
    return pd.DataFrame(rows)

promotions_df = gen_promotions()
print(f"Promotions generated: {len(promotions_df)}")

# ---------------------------------------------------------------------------
# 3. CUSTOMERS (~5500, weighted join dates, behavior segments)
# ---------------------------------------------------------------------------
N_CUSTOMERS = 5500
YEAR_WEIGHTS = {2023: 0.15, 2024: 0.25, 2025: 0.30, 2026: 0.30}
SEGMENTS = ["one_time", "occasional", "regular", "loyal"]
SEGMENT_WEIGHTS = [0.25, 0.35, 0.25, 0.15]

def random_date_in_year(year):
    yr_start = date(year, 1, 1)
    yr_end = date(year, 12, 31) if year < 2026 else END_DATE
    yr_start = max(yr_start, START_DATE)
    delta = (yr_end - yr_start).days
    return yr_start + timedelta(days=random.randint(0, max(delta, 0)))

def gen_customers():
    rows = []
    for cid in range(1, N_CUSTOMERS + 1):
        year = random.choices(list(YEAR_WEIGHTS.keys()), weights=list(YEAR_WEIGHTS.values()))[0]
        join_date = random_date_in_year(year)
        segment = random.choices(SEGMENTS, weights=SEGMENT_WEIGHTS)[0]
        dob = fake.date_of_birth(minimum_age=18, maximum_age=55)
        first = fake.first_name()
        last = fake.last_name()
        state = random.choice(INDIAN_STATES)
        rows.append({
            "customer_id": cid,
            "first_name": first,
            "last_name": last,
            "email": f"{first.lower()}.{last.lower()}{cid}@{fake.free_email_domain()}",
            "phone_number": fake.msisdn()[:10],
            "date_of_birth": dob.isoformat(),
            "address": fake.address().replace("\n", ", "),
            "state": state,
            "country": "India",
            "join_date": join_date.isoformat(),
            "_segment": segment,
        })
    return pd.DataFrame(rows)

customers_df = gen_customers()
print(f"Customers generated: {len(customers_df)}")
print(customers_df["_segment"].value_counts())

# ---------------------------------------------------------------------------
# 4. ORDERS + ORDER_DETAILS (segment-driven frequency, churn, festival spikes)
# ---------------------------------------------------------------------------
SEGMENT_ORDER_RANGE = {
    "one_time": (1, 1),
    "occasional": (2, 6),
    "regular": (8, 20),
    "loyal": (20, 42),
}
# fraction of regular/loyal customers who churn (stop ordering well before END_DATE)
CHURN_RATE = {"one_time": 0.0, "occasional": 0.35, "regular": 0.30, "loyal": 0.22}

# festival windows for order-date clustering (month-day, spread days, weight multiplier)
FESTIVAL_WINDOWS = [(1, 1, 7), (1, 26, 5), (3, 15, 6), (6, 15, 14),
                     (8, 15, 5), (8, 25, 4), (10, 5, 9), (11, 1, 10), (12, 20, 12)]

def festival_dates_in_range(start, end):
    dates = []
    for year in range(start.year, end.year + 1):
        for month, day, spread in FESTIVAL_WINDOWS:
            try:
                base = date(year, month, day)
            except ValueError:
                continue
            if start <= base <= end:
                dates.append((base, spread))
    return dates

def pick_order_date(cust_join, window_end):
    """70% chance cluster near a festival window within the customer's active period, else random."""
    fests = festival_dates_in_range(cust_join, window_end)
    if fests and random.random() < 0.4:
        base, spread = random.choice(fests)
        offset = random.randint(-spread, spread)
        d = base + timedelta(days=offset)
        d = max(cust_join, min(d, window_end))
        return d
    span = (window_end - cust_join).days
    if span <= 0:
        return cust_join
    return cust_join + timedelta(days=random.randint(0, span))

promo_lookup = promotions_df[["promotion_id", "start_date", "end_date"]].copy()
promo_lookup["start_date"] = pd.to_datetime(promo_lookup["start_date"])
promo_lookup["end_date"] = pd.to_datetime(promo_lookup["end_date"])

def promo_active_on(d):
    d_ts = pd.Timestamp(d)
    active = promo_lookup[(promo_lookup["start_date"] <= d_ts) & (promo_lookup["end_date"] >= d_ts)]
    if len(active) == 0:
        return None
    return int(active.sample(1)["promotion_id"].values[0])

products_list = products_df.to_dict("records")
category_products = {}
for p in products_list:
    category_products.setdefault(p["category"], []).append(p)

order_rows = []
order_details_rows = []
order_id_counter = 1
od_id_counter = 1
END_TS = END_DATE

for _, cust in customers_df.iterrows():
    seg = cust["_segment"]
    join_d = date.fromisoformat(cust["join_date"])
    low, high = SEGMENT_ORDER_RANGE[seg]
    n_orders = random.randint(low, high)

    # determine customer's active window (churn simulation)
    will_churn = random.random() < CHURN_RATE.get(seg, 0)
    if will_churn and seg != "one_time":
        # active window ends somewhere between 60 and 500 days before END_DATE
        churn_gap = random.randint(60, 500)
        active_end = min(END_DATE - timedelta(days=churn_gap), END_DATE)
        active_end = max(active_end, join_d + timedelta(days=1))
    else:
        active_end = END_DATE

    order_dates = sorted([pick_order_date(join_d, active_end) for _ in range(n_orders)])

    for od in order_dates:
        oid = order_id_counter
        order_id_counter += 1

        use_promo = random.random() < 0.30  # 70:30 no-promo:promo ratio (target, enforced below)
        promo_id = promo_active_on(od) if use_promo else None
        if use_promo and promo_id is None:
            # no festival active that exact day - still honor the 30% ratio by assigning
            # any promotion (simplification: treated as a generic/evergreen offer applied at checkout)
            promo_id = int(promotions_df.sample(1)["promotion_id"].values[0])

        # line items for this order
        n_lines = random.choices([1, 2, 3, 4], weights=[45, 30, 15, 10])[0]
        cat_choice = random.choice(list(category_products.keys()))
        chosen_products = random.sample(category_products[cat_choice],
                                         min(n_lines, len(category_products[cat_choice])))
        if len(chosen_products) < n_lines:
            extra_pool = [p for p in products_list if p not in chosen_products]
            chosen_products += random.sample(extra_pool, n_lines - len(chosen_products))

        line_total_sum = 0.0
        order_detail_ids_this_order = []
        for prod in chosen_products:
            qty = random.choices([1, 2, 3], weights=[70, 22, 8])[0]
            unit_price = prod["selling_price"]
            total_price = round(qty * unit_price, 2)
            order_details_rows.append({
                "order_details_id": od_id_counter,
                "order_id": oid,
                "product_id": prod["product_id"],
                "quantity": qty,
                "unit_price": unit_price,
                "total_price": total_price,
                "_category": prod["category"],
                "_return_rate": prod["_return_rate"],
                "_order_date": od,
            })
            order_detail_ids_this_order.append(od_id_counter)
            od_id_counter += 1
            line_total_sum += total_price

        discount_pct = 0.0
        if use_promo and promo_id is not None:
            discount_pct = float(promotions_df.loc[promotions_df.promotion_id == promo_id, "discount_percentage"].values[0])
        discount_amount = round(line_total_sum * discount_pct / 100, 2)
        shipping_charge = 0.0 if line_total_sum >= 999 else round(random.uniform(40, 99), 2)
        order_total_amount = round(line_total_sum - discount_amount + shipping_charge, 2)

        status = random.choices(
            ["Delivered", "Cancelled", "Returned_Partial_Flow"],  # returns handled via order_details separately
            weights=[92, 5, 3]
        )[0]
        # simplify: treat "Returned_Partial_Flow" as Delivered (return tracked at line level in Returns table)
        order_status = "Delivered" if status != "Cancelled" else "Cancelled"

        order_rows.append({
            "order_id": oid,
            "customer_id": cust["customer_id"],
            "promotion_id": promo_id if use_promo else None,
            "order_date": od.isoformat(),
            "order_status": order_status,
            "shipping_address": cust["address"],
            "billing_address": cust["address"],
            "discount_amount": discount_amount,
            "shipping_charge": shipping_charge,
            "order_total_amount": order_total_amount,
        })

orders_df = pd.DataFrame(order_rows)
order_details_df = pd.DataFrame(order_details_rows)
print(f"Orders generated: {len(orders_df)}")
print(f"Order details generated: {len(order_details_df)}")

# ---------------------------------------------------------------------------
# 5. RETURNS (category-based rates, only on Delivered orders)
# ---------------------------------------------------------------------------
delivered_order_ids = set(orders_df.loc[orders_df.order_status == "Delivered", "order_id"])
return_reasons_by_cat = {
    "Clothing": ["Size too small", "Size too large", "Fabric quality poor", "Color mismatch", "Changed mind"],
    "Footwear": ["Wrong size", "Uncomfortable fit", "Sole damage", "Color mismatch", "Changed mind"],
    "Hats": ["Size issue", "Not as described", "Changed mind"],
    "Makeup": ["Shade mismatch", "Product damaged", "Allergic reaction", "Changed mind"],
    "Soap": ["Product damaged", "Not as described", "Changed mind"],
}

return_rows = []
return_id_counter = 1
for _, line in order_details_df.iterrows():
    if line["order_id"] not in delivered_order_ids:
        continue
    if random.random() < line["_return_rate"]:
        ret_date = line["_order_date"] + timedelta(days=random.randint(2, 15))
        if ret_date > END_DATE:
            continue
        return_rows.append({
            "return_id": return_id_counter,
            "order_details_id": line["order_details_id"],
            "return_reason": random.choice(return_reasons_by_cat[line["_category"]]),
            "return_amount": line["total_price"],
            "return_status": random.choices(["Approved", "Rejected", "Pending"], weights=[80, 10, 10])[0],
            "return_date": ret_date.isoformat(),
        })
        return_id_counter += 1

returns_df = pd.DataFrame(return_rows)
print(f"Returns generated: {len(returns_df)}")

# ---------------------------------------------------------------------------
# 6. REVIEWS (subset of delivered, valid purchase lines)
# ---------------------------------------------------------------------------
REVIEW_TEMPLATES_POS = [
    "Really happy with this purchase, great quality!",
    "Exactly as described, will buy again.",
    "Good value for money.",
    "Loved it, fits perfectly.",
    "Nice product, fast delivery too.",
]
REVIEW_TEMPLATES_NEU = [
    "It's okay, expected a bit better.",
    "Decent product for the price.",
    "Average quality, does the job.",
]
REVIEW_TEMPLATES_NEG = [
    "Not satisfied with the quality.",
    "Product did not match the description.",
    "Disappointed, expected better.",
]

review_rows = []
review_id_counter = 1
eligible_lines = order_details_df[order_details_df.order_id.isin(delivered_order_ids)]
review_sample = eligible_lines.sample(min(35000, len(eligible_lines)), random_state=42)

for _, line in review_sample.iterrows():
    rating = random.choices([5, 4, 3, 2, 1], weights=[38, 30, 18, 9, 5])[0]
    if rating >= 4:
        text = random.choice(REVIEW_TEMPLATES_POS)
    elif rating == 3:
        text = random.choice(REVIEW_TEMPLATES_NEU)
    else:
        text = random.choice(REVIEW_TEMPLATES_NEG)
    rev_date = line["_order_date"] + timedelta(days=random.randint(3, 25))
    if rev_date > END_DATE:
        continue
    review_rows.append({
        "review_id": review_id_counter,
        "order_details_id": line["order_details_id"],
        "rating": rating,
        "review_text": text,
        "review_date": rev_date.isoformat(),
    })
    review_id_counter += 1

reviews_df = pd.DataFrame(review_rows)
print(f"Reviews generated: {len(reviews_df)}")

print("\n--- RAW GENERATION COMPLETE (clean stage) ---")
print(f"Total rows across all tables: "
      f"{len(customers_df)+len(products_df)+len(promotions_df)+len(orders_df)+len(order_details_df)+len(returns_df)+len(reviews_df)}")

# drop internal helper columns before messiness/export
products_export = products_df.drop(columns=["_return_rate"]).copy()
customers_export = customers_df.drop(columns=["_segment"]).copy()
order_details_export = order_details_df.drop(columns=["_category", "_return_rate", "_order_date"]).copy()

# ---------------------------------------------------------------------------
# 7. MESSINESS INJECTION (~5-7% of rows affected, realistic issue types)
# ---------------------------------------------------------------------------
MESS_RATE = 0.06

def blank_out(df, col, rate=MESS_RATE):
    n = int(len(df) * rate)
    idx = df.sample(n, random_state=random.randint(1, 99999)).index
    df.loc[idx, col] = np.nan
    return df

def messy_case(val):
    if pd.isna(val):
        return val
    choice = random.random()
    if choice < 0.33:
        return val.upper()
    elif choice < 0.66:
        return val.lower()
    return val

def messy_date_format(iso_str):
    if pd.isna(iso_str):
        return iso_str
    d = datetime.fromisoformat(iso_str)
    fmt = random.choice(["%d-%m-%Y", "%m/%d/%Y", "%Y/%m/%d", "%d %b %Y"])
    return d.strftime(fmt)

def messy_phone(p):
    if pd.isna(p):
        return p
    style = random.random()
    if style < 0.25:
        return f"+91-{p}"
    elif style < 0.5:
        return f"{p[:5]}-{p[5:]}"
    elif style < 0.65:
        return f"0{p}"
    return p

# --- Customers messiness ---
customers_export = blank_out(customers_export, "email", 0.04)
customers_export = blank_out(customers_export, "phone_number", 0.03)
customers_export = blank_out(customers_export, "address", 0.02)

idx = customers_export.sample(int(len(customers_export) * 0.05), random_state=7).index
customers_export.loc[idx, "state"] = customers_export.loc[idx, "state"].apply(messy_case)

idx = customers_export.sample(int(len(customers_export) * 0.05), random_state=8).index
customers_export.loc[idx, "phone_number"] = customers_export.loc[idx, "phone_number"].apply(messy_phone)

idx = customers_export.sample(int(len(customers_export) * 0.04), random_state=9).index
customers_export.loc[idx, "join_date"] = customers_export.loc[idx, "join_date"].apply(messy_date_format)

# duplicate ~1% of customers (same person, new customer_id -> realistic re-registration issue)
n_dupes = int(len(customers_export) * 0.01)
dupes = customers_export.sample(n_dupes, random_state=11).copy()
dupes["customer_id"] = range(customers_export["customer_id"].max() + 1,
                              customers_export["customer_id"].max() + 1 + n_dupes)
customers_export = pd.concat([customers_export, dupes], ignore_index=True)

# whitespace issues in names
idx = customers_export.sample(int(len(customers_export) * 0.03), random_state=12).index
customers_export.loc[idx, "first_name"] = "  " + customers_export.loc[idx, "first_name"] + "  "

# --- Orders messiness ---
idx = orders_df.sample(int(len(orders_df) * 0.04), random_state=13).index
orders_df.loc[idx, "order_date"] = orders_df.loc[idx, "order_date"].apply(messy_date_format)

idx = orders_df.sample(int(len(orders_df) * 0.05), random_state=14).index
orders_df.loc[idx, "order_status"] = orders_df.loc[idx, "order_status"].apply(messy_case)

idx = orders_df.sample(int(len(orders_df) * 0.02), random_state=15).index
orders_df.loc[idx, "shipping_address"] = np.nan

# intentional total mismatches (~1.5% of orders) -- ties to Phase 3 validation rule
idx = orders_df.sample(int(len(orders_df) * 0.015), random_state=16).index
orders_df.loc[idx, "order_total_amount"] = orders_df.loc[idx, "order_total_amount"] + \
    np.random.choice([-50, -20, 15, 30, 100], size=len(idx))

# --- Order_Details messiness ---
idx = order_details_export.sample(int(len(order_details_export) * 0.01), random_state=17).index
order_details_export.loc[idx, "quantity"] = order_details_export.loc[idx, "quantity"].apply(
    lambda q: random.choice([0, -1]))

idx = order_details_export.sample(int(len(order_details_export) * 0.03), random_state=18).index
order_details_export.loc[idx, "unit_price"] = np.nan

# --- Products messiness ---
idx = products_export.sample(int(len(products_export) * 0.06), random_state=19).index
products_export.loc[idx, "category"] = products_export.loc[idx, "category"].apply(messy_case)

idx = products_export.sample(int(len(products_export) * 0.04), random_state=20).index
products_export.loc[idx, "launch_date"] = products_export.loc[idx, "launch_date"].apply(messy_date_format)

# --- Returns messiness ---
idx = returns_df.sample(int(len(returns_df) * 0.05), random_state=21).index
returns_df.loc[idx, "return_reason"] = returns_df.loc[idx, "return_reason"].apply(messy_case)

idx = returns_df.sample(int(len(returns_df) * 0.03), random_state=22).index
returns_df.loc[idx, "return_date"] = returns_df.loc[idx, "return_date"].apply(messy_date_format)

# --- Reviews messiness ---
idx = reviews_df.sample(int(len(reviews_df) * 0.03), random_state=23).index
reviews_df.loc[idx, "review_text"] = np.nan

idx = reviews_df.sample(int(len(reviews_df) * 0.02), random_state=24).index
reviews_df.loc[idx, "rating"] = reviews_df.loc[idx, "rating"].apply(lambda r: random.choice([0, 6, -1]))

# ---------------------------------------------------------------------------
# 8. EXPORT TO CSV
# ---------------------------------------------------------------------------
import os
OUT_DIR = "/home/claude/project/veloura_raw_data"
os.makedirs(OUT_DIR, exist_ok=True)

customers_export.to_csv(f"{OUT_DIR}/customers.csv", index=False)
products_export.to_csv(f"{OUT_DIR}/products.csv", index=False)
promotions_df.to_csv(f"{OUT_DIR}/promotions.csv", index=False)
orders_df.to_csv(f"{OUT_DIR}/orders.csv", index=False)
order_details_export.to_csv(f"{OUT_DIR}/order_details.csv", index=False)
returns_df.to_csv(f"{OUT_DIR}/returns.csv", index=False)
reviews_df.to_csv(f"{OUT_DIR}/reviews.csv", index=False)

print("\n--- MESSY DATA EXPORTED ---")
for f in ["customers", "products", "promotions", "orders", "order_details", "returns", "reviews"]:
    path = f"{OUT_DIR}/{f}.csv"
    print(f"{f}.csv -> {sum(1 for _ in open(path)) - 1} rows")

total_final = (len(customers_export) + len(products_export) + len(promotions_df) +
               len(orders_df) + len(order_details_export) + len(returns_df) + len(reviews_df))
print(f"\nFINAL TOTAL ROWS (all tables combined): {total_final}")


Products generated: 55
Promotions generated: 100
Customers generated: 5500
_segment
occasional    1897
one_time      1420
regular       1397
loyal          786
Name: count, dtype: int64
Orders generated: 52855
Order details generated: 100273
Returns generated: 10941
Reviews generated: 33137

--- RAW GENERATION COMPLETE (clean stage) ---
Total rows across all tables: 202861

--- MESSY DATA EXPORTED ---
customers.csv -> 5555 rows
products.csv -> 55 rows
promotions.csv -> 100 rows
orders.csv -> 52855 rows
order_details.csv -> 100273 rows
returns.csv -> 10941 rows
reviews.csv -> 33137 rows

FINAL TOTAL ROWS (all tables combined): 202916


In [2]:
pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 23.7 MB/s eta 0:00:00
